# 03: データセットの確認と再生

Hugging Faceで公開されているLeRobot版のLIBEROから、選んだエピソードをColabで読み込みます。カメラ画像・状態・行動が、同じ時刻にどう対応するかを確認します。データの版は `dataset.json` の `revision` で固定しています。

[取得元のデータセット（Hugging Face）](https://huggingface.co/datasets/lerobot/libero) · [実習で使う版](https://huggingface.co/datasets/lerobot/libero/tree/a1aaacb7f6cd6ee5fb43120f673cebb0cfea7dd4)

1. Google Driveの「マイドライブ」に `vla_workshop` フォルダを作り、その中に `03_dataset` をフォルダごと置きます。
2. このノートブックをColabで開き、ランタイムを **CPU（ハードウェアアクセラレータなし）** にします。別の場所に置いた場合は、最初のコードセルの `LESSON_DIR` を変更します。
3. 上から順にセルを実行します。

同じフォルダに置くファイル

- `dataset.json`
- `prepare.py`
- `requirements.txt`

In [ ]:
# @title この回の配布ファイルを準備する
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive, files

USE_DRIVE = True  # @param {type:"boolean"}
LESSON_DIR = "/content/drive/MyDrive/vla_workshop/03_dataset"  # @param {type:"string"}
REQUIRED_FILES = [
    "dataset.json",
    "prepare.py",
    "requirements.txt",
]

if USE_DRIVE:
    drive.mount("/content/drive")
    source = Path(LESSON_DIR)
else:
    # ipynb以外の今回の配布ファイルを、まとめて選んでアップロードします。
    os.chdir("/content")
    files.upload()
    source = Path("/content")
missing = [name for name in REQUIRED_FILES if not (source / name).is_file()]
if missing:
    raise FileNotFoundError(f"配布ファイルが不足: {missing}。LESSON_DIRを確認してください。")
PROJECT = Path("/content/lesson_03")
PROJECT.mkdir(exist_ok=True)
for name in REQUIRED_FILES:
    shutil.copy2(source / name, PROJECT / name)
os.chdir(PROJECT)
print("この回の作業場所:", PROJECT)

## ライブラリの準備

次のセルで、この回に必要なライブラリを準備します。初回は数分かかることがあります。

In [ ]:
# @title ライブラリを準備する
LESSON_ENV = {
    **os.environ,
    "MPLBACKEND": "Agg",
    "PYTHONPATH": str(PROJECT),
    "PYTHONNOUSERSITE": "1",
}
command = [sys.executable, str(PROJECT / "prepare.py")]
subprocess.run(command, env=LESSON_ENV, check=True)
PYTHON = PROJECT / ".venv/bin/python"


def run_python(program):
    """セルに書いたPythonを、この回の専用Pythonで実行する。"""
    last_result = None
    with subprocess.Popen(
        [str(PYTHON), "-u", "-c", program],
        cwd=PROJECT,
        env=LESSON_ENV,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            if line.startswith("Results: "):
                last_result = Path(line.removeprefix("Results: ").strip())
        if process.wait():
            raise RuntimeError("実行に失敗しました。直前のエラーを確認してください。")
    return last_result

## 再生するエピソードIDを選ぶ

エピソードは、ロボットの動作1回分の記録です。ここで見る4件は、02のLIBERO-Spatialのタスク0と同じ指示で集められた記録です。`REPLAY_EPISODE` でIDを1つ選び、このセルから下を実行してください。

| ID | 04での使い方 |
| ---: | --- |
| 1642、1660、1669 | LoRAの学習に使う |
| 1672 | 検証lossの確認に使う |

別の記録を見たいときは、IDを変えてこのセルからもう一度実行します。

In [ ]:
# @title 再生するエピソードIDを選ぶ
REPLAY_EPISODE = "1642"  # @param ["1642", "1660", "1669", "1672"]

config = json.loads((PROJECT / "dataset.json").read_text())
train = config["train_episodes"]
validation = config["validation_episodes"]
episode = int(REPLAY_EPISODE)
if set(train) & set(validation):
    raise ValueError("学習と検証のエピソードを重複させないでください")
if episode not in train + validation:
    raise ValueError("再生するエピソードは学習用または検証用から選んでください")

config["replay_episode"] = episode
purpose = "学習用" if episode in train else "検証用"
print(f"再生するエピソードID: {episode}（{purpose}）")
(PROJECT / "active_dataset.json").write_text(json.dumps(config))

## 指定したエピソードを読み込んで再生する

`LeRobotDataset` にデータセット名とエピソードIDを渡し、Hugging Faceから取得します。2視点の映像を横に並べ、冒頭から最大100フレームを再生します。

In [ ]:
result = run_python(r"""
import json
import tempfile
from pathlib import Path

import imageio.v2 as imageio
import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset

config = json.loads(Path("active_dataset.json").read_text())
episode = config["replay_episode"]
# エピソードを選び直せるように、取得したファイルはIDごとに保存します。
root = Path("datasets") / config["repo_id"] / config["revision"] / str(episode)
dataset = LeRobotDataset(
    config["repo_id"],
    root=root,
    revision=config["revision"],
    episodes=[episode],
    video_backend="pyav",
)
frame = dataset[0]
report = {
    "episode": episode,
    "instruction": frame["task"],
    "frames": len(dataset),
    "fps": dataset.fps,
    "state": frame["observation.state"].tolist(),
    "action": frame["action"].tolist(),
}
folder = Path(tempfile.mkdtemp(prefix="dataset-", dir="outputs")).resolve()
(folder / "dataset.json").write_text(json.dumps(report, indent=2, ensure_ascii=False))

cameras = ("observation.images.image", "observation.images.image2")
with imageio.get_writer(folder / "replay.mp4", fps=dataset.fps) as video:
    for index in range(min(len(dataset), config["preview_frames"])):
        frame = dataset[index]
        # CHWからHWCへ変換し、2視点の画像を横に並べます。
        images = [frame[key].permute(1, 2, 0).numpy() for key in cameras]
        rgb = np.concatenate(images, axis=1)
        video.append_data(np.clip(rgb * 255, 0, 255).astype(np.uint8))
print(f"Results: {folder}")
""")

## 2視点の動画と、最初のフレームの状態・行動を見る

In [ ]:
from IPython.display import Video, display

report = json.loads((result / "dataset.json").read_text())
print("エピソードID:", report["episode"])
print("指示:", report["instruction"])
print(f"記録の長さ: {report['frames']}フレーム（{report['fps']} fps）")
print("最初のフレームのstate:", report["state"])
print("最初のフレームのaction:", report["action"])
display(Video(str(result / "replay.mp4"), embed=True))

## 動画で見ておくこと

動画の最初のフレームと、そのときの `state`・`action` を確認してください。別のエピソードIDを選び、同じ指示でもロボットの動きにどんな違いがあるか見てみましょう。

## 結果をDriveに保存する

データの確認結果と再生動画を保存します。

In [ ]:
# @title 結果を保存する
from datetime import UTC, datetime
from uuid import uuid4

RESULTS_DIR = "/content/drive/MyDrive/vla_workshop/results"  # @param {type:"string"}
drive.mount("/content/drive")
destination = Path(RESULTS_DIR)
destination.mkdir(parents=True, exist_ok=True)
name = "03_dataset_" + datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
saved = destination / name
shutil.copytree(PROJECT / "outputs", saved)
print("保存先:", saved)